## Script for histograms creation

In [31]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import time
import os
import hashlib
import yaml


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression, save_resolutions_to_yaml
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT(90)
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

print(channName)

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [2]:
base_path = "/workspace/root_notebooks/results"

################ SETTINGS ##################################
analysis_dir = "signal_separated_kin_fits"
img_dir = "img"
date = "2026-10-28"
root_file_name = "separated_kin_fits_1D_truth_variables.root"
eff_graph_file_name = "efficiency_purity_graphs.root"

mode_chann = "channels"

if mode_chann == "channels":
  flags = channName.values()
else:
  flags = ["has000", "no000", "Data", "MC sum"]
  
mode = "zoom"
nocut_hash = "nocuts"
log_flag = False
fit_double_gauss = True

hist_config_file_path = f"{base_path}/{analysis_dir}/config/histograms_config_1D.yaml"
with open(hist_config_file_path, "r") as f:
    hist_config = yaml.safe_load(f)
############################################################

hist_file_path = f"{base_path}/{analysis_dir}/{date}/{root_file_name}"
eff_graph_file_path = f"{base_path}/{analysis_dir}/{date}/{eff_graph_file_name}"

#################### ANALYSIS REGISTRY #####################
import pandas as pd
analysis_registry = f"{base_path}/{analysis_dir}/{date}/analysis_registry.csv"
df = pd.read_csv(analysis_registry)

hash_available = df["hash"]

## Making histograms

In [3]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)
ROOT.TH1.AddDirectory(False)

# Set up variables 1D
variables_1D_truth = [hist for hist in hist_config.get("truth_only_vars_1d", [])]

#################### OBJECT TO FIND ################################
histograms = {}

for hash in hash_available:
    if hash not in [key.GetName() for key in hist_file.GetListOfKeys()]:
        continue
    
    histograms[hash] = {}

    for var in variables_1D_truth:
      dir_obj = hist_file.Get(f"{hash}/{var}")
      if var not in histograms[hash]:
          histograms[hash][var] = {}

      for flag in flags:    
        hist_name = f"{var}_{flag}_{mode}"
        if hist_name not in [key.GetName() for key in dir_obj.GetListOfKeys()]:
            continue
    
        histograms[hash][var][flag] = dir_obj.Get(hist_name).Clone()

hist_file.Close()


In [33]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

################## DRAW USING THSTACK ###########################
canvases = []
legends = []
stacks = []
funcs = []
par_dict = {}

for hash in histograms:    
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)
    for i, var in enumerate(histograms[hash]):
        path_var = f"{path_hash}/{var}"
        os.makedirs(path_var, exist_ok=True)
        
        legend = ROOT.TLegend(0.5, 0.8, 0.84875, 0.9)
        legend.SetTextFont(62)
        legend.SetTextSize(0.025)
        
        # Dodajemy hash do nazwy, aby uniknąć konfliktów w pamięci ROOTa przy wielu folderach
        stack = ROOT.THStack(f"{var}_stack_{hash}", "")
        
        for flag in histograms[hash][var]:
            hist = histograms[hash][var][flag]
            if hist:

              if fit_double_gauss:
                fit_func = ROOT.TF1(f"{var}_fit_{hash}_{flag}", ROOT.double_gaussian_cpp, hist.GetXaxis().GetXmin(), hist.GetXaxis().GetXmax(), 5)
                fit_func.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS() * 0.5, hist.GetRMS() * 2.0, 0.7)
                fit_func.SetParNames("A1", "mean", "sigma1", "sigma2", "f")
                fit_func.SetParLimits(0, 0, hist.GetMaximum() * 50)
                fit_func.SetParLimits(1, hist.GetMean() - hist.GetRMS(), hist.GetMean() + hist.GetRMS())
                fit_func.SetParLimits(2, 0, hist.GetRMS() * 2)
                fit_func.SetParLimits(3, 0, hist.GetRMS() * 4)
                fit_func.SetParLimits(4, 0, 1)

                fit_func.SetLineColor(ROOT.kBlack)
                fit_func.SetLineStyle(ROOT.kDashed)
                
                fit_result = hist.Fit(fit_func, "ISR")

                if fit_result.Status() == 0:
                  funcs.append(fit_func)


              hist.SetLineColor(channColor[flag])

              if log_flag:
                hist.SetMinimum(0.1)
                
              if flag != "Data":
                # Opcję rysowania przekazujemy bezpośrednio w metodzie Add dla THStack
                stack.Add(hist, "hist")
                legend.AddEntry(hist, channTitle[flag], "l")

                if fit_double_gauss:
                  legend.AddEntry(fit_func, "Double Gaussian Fit", "l")

              else:
                hist.SetMarkerStyle(20)
                hist.SetMarkerColor(channColor[flag])
                stack.Add(hist, "pe1")
                legend.AddEntry(hist, channTitle[flag], "pe1")
                            
        canvas = ROOT.TCanvas(f"{var}_canvas_{hash}", f"{var} Canvas", 600, 600)
        
        stack.Draw("NOSTACK")
        if fit_double_gauss:
            fit_func.Draw("same")

            pave = ROOT.TPaveText(0.15, 0.55, 0.43, 0.9, "NDC")

            # Konfiguracja wyglądu pudełka (opcjonalnie)
            pave.SetFillColor(0)       # Białe tło (0 = przezroczyste/białe w zależności od stylu)
            pave.SetBorderSize(1)      # Cienka ramka (0 = brak ramki)
            pave.SetTextFont(42)       # Zwykła czcionka (lub 62 dla bolda)
            pave.SetTextSize(0.023)
            pave.SetTextAlign(12)      # Wyrównanie tekstu: 1=lewa, 2=środek w pionie

            # 2. Dodajemy kolejne linijki tekstu
            pave.AddText("Double Gaussian Fit")

            if int(fit_result) == 0:

                mean, mean_err = fit_result.Parameter(1), fit_result.ParError(1)
                sigma_core, sigma_core_err = fit_result.Parameter(2), fit_result.ParError(2)
                sigma_tail, sigma_tail_err = fit_result.Parameter(3), fit_result.ParError(3)
                amplitude_ratio, amplitude_ratio_err = fit_result.Parameter(4), fit_result.ParError(4)
                chi2_ndf = fit_result.Chi2() / fit_result.Ndf() if fit_result.Ndf() != 0 else float('nan')

                real_var = var.removesuffix("_resolution")

                par_dict[real_var] = {
                    "mean": [mean, mean_err],
                    "sigma_core": [sigma_core, sigma_core_err],
                    "sigma_tail": [sigma_tail, sigma_tail_err],
                    "amplitude_ratio": [amplitude_ratio, amplitude_ratio_err],
                    "fit_status": int(fit_result),
                    "chi2_ndf": chi2_ndf,
                }

                pave.AddText(f'A_{{core}} = {fit_result.Parameter(0):.2g} #pm {fit_result.ParError(0):.2g}')
                pave.AddText(f'#mu = ({fit_result.Parameter(1):.2g} #pm {fit_result.ParError(1):.2g}) {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}')
                pave.AddText(f'#sigma_{{core}} = ({fit_result.Parameter(2):.2g} #pm {fit_result.ParError(2):.2g}) {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}')
                pave.AddText(f'#sigma_{{tail}} = ({fit_result.Parameter(3):.2g} #pm {fit_result.ParError(3):.2g}) {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}')
                pave.AddText(f'#frac{{A_{{core}}}}{{A_{{tail}}}} = {fit_result.Parameter(4):.2g} #pm {fit_result.ParError(4):.2g}')
            else:
                pave.AddText("Fit failed")

            # 3. Rysujemy całe pudełko na płótnie jednym poleceniem!
            pave.Draw()
        
        if stack.GetHistogram():
            resolution = stack.GetHistogram().GetXaxis().GetBinWidth(1)

            title_x = f'{hist_config.get("histograms_1d", {}).get(var, {}).get("title", "")} [{hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}]'
            title_y = f'Counts / {resolution:.2f} {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}'
            
            stack.GetHistogram().GetXaxis().SetTitle(title_x)
            stack.GetHistogram().GetYaxis().SetTitle(title_y)

        ####### Set log y ########
        if log_flag:
          stack.SetMinimum(0.1)
          stack.SetMaximum(stack.GetMaximum() * 50)
        else:
          stack.SetMinimum(0)
                
        canvas.SetLogy(log_flag)
        ##########################

        legend.Draw()

        canvas.Draw()
        canvas.SaveAs(f"{path_var}/{var}_{mode}_logy_{log_flag}.pdf")

    save_resolutions_to_yaml(par_dict, f"{path_hash}/resolutions_config.yaml")

        

Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/nocuts/resolutions_config.yaml
Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/26f97643/resolutions_config.yaml
Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/17dde94b/resolutions_config.yaml
Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/a0d936a9/resolutions_config.yaml
Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/ec4393ae/resolutions_config.yaml
Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/62d706c5/resolutions_config.yaml
Pomyślnie zrzucono wyniki rozdzielczości do plik

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/nocuts/Kchrec_pz_resolution/Kchrec_pz_resolution_zoom_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/nocuts/Kchrec_y_resolution/Kchrec_y_resolution_zoom_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/nocuts/ip_bhabha_z_resolution/ip_bhabha_z_resolution_zoom_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/nocuts/KnerecFit_z_resolution/KnerecFit_z_resolution_zoom_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/signal_separated_kin_fits/2026-10-28/img/nocuts/Delta_t_rec_resolution/Delta_t_rec_resolution_zoom_logy_False.pdf has been created
Info in <T

In [ ]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)
ROOT.TH1.AddDirectory(False)

# Set up variables 1D
variables_1D_truth = [hist for hist in hist_config.get("truth_only_vars_1d", [])]

#################### OBJECT TO FIND ################################
histograms = {}

for hash in hash_available:
    if hash not in [key.GetName() for key in hist_file.GetListOfKeys()]:
        continue
    
    histograms[hash] = {}

    for var in variables_1D_truth:
      dir_obj = hist_file.Get(f"{hash}/{var}")
      if var not in histograms[hash]:
          histograms[hash][var] = {}

      for flag in flags:    
        hist_name = f"{var}_{flag}_{mode}"
        if hist_name not in [key.GetName() for key in dir_obj.GetListOfKeys()]:
            continue
    
        histograms[hash][var][flag] = dir_obj.Get(hist_name).Clone()

hist_file.Close()


## Calculating events, efficiencies and purities

In [42]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)

#################### OBJECT TO FIND ################################
event_counts = {}
cuts_list = {}
efficiencies = {}
purities = {}

#################### EXTRACT EVENT COUNTS #######################
for hash in hash_available:
    obj = hist_file.Get(f"{hash}/ChannelIntegrals")
    obj_cut_list = hist_file.Get(f"{hash}/CutList")

    if obj:
      chann_integrals_str = obj.GetTitle()
      event_counts[hash] = json.loads(chann_integrals_str)

    if obj_cut_list:
      raw_title = obj_cut_list.GetTitle()
      cleaned_str = raw_title.strip("[] ").replace('"', '')
      cuts_list[hash] = [c.strip() for c in cleaned_str.split(',') if c.strip()]

hist_file.Close()
#################################################################

#################### CALCULATE EFFICIENCIES AND PURITIES #######################
for hash in hash_available:
  if hash not in event_counts:
     continue

  efficiencies[hash] = {}
  purities[hash] = {}

  mc_sum = int(event_counts[hash].get("MC sum", 0))

  for flag in flags:
      total_events = int(event_counts.get(nocut_hash, None).get(flag, 0))
      passed_events = int(event_counts[hash].get(flag, 0))
      efficiencies[hash][flag] = passed_events / total_events if total_events > 0 else 0
      purities[hash][flag] = passed_events / mc_sum if mc_sum > 0 else 0
###############################################################################

################## PRINT RESULTS ##################
for hash in hash_available:
    if hash not in efficiencies:
        continue
    print(f"Results for cut list: {cuts_list.get(hash, [])}")
    for flag in flags:
        event_count = int(event_counts.get(hash, {}).get(flag, 0))
        mc_sum = int(event_counts.get(hash, {}).get("MC sum", 0))
        eff = efficiencies[hash].get(flag, 0)
        pur = purities[hash].get(flag, 0)
        print(f"  Flag: {flag}, Event Count: {event_count / 10**6:.2f} x 10^6, Efficiency: {eff * 100:.2f}%, Purity: {pur * 100:.2f}%, (MC - Data) / Data: {100*(mc_sum - event_count) / event_count if event_count > 0 else 0:.2f}%")
    print()




Results for cut list: []
  Flag: Data, Event Count: 0.00 x 10^6, Efficiency: 0.00%, Purity: 0.00%, (MC - Data) / Data: 0.00%
  Flag: Signal, Event Count: 0.43 x 10^6, Efficiency: 100.00%, Purity: 0.00%, (MC - Data) / Data: -100.00%
  Flag: Regeneration, Event Count: 0.00 x 10^6, Efficiency: 0.00%, Purity: 0.00%, (MC - Data) / Data: 0.00%
  Flag: Omega, Event Count: 0.00 x 10^6, Efficiency: 0.00%, Purity: 0.00%, (MC - Data) / Data: 0.00%
  Flag: 3pi0, Event Count: 0.00 x 10^6, Efficiency: 0.00%, Purity: 0.00%, (MC - Data) / Data: 0.00%
  Flag: Semileptonic, Event Count: 0.00 x 10^6, Efficiency: 0.00%, Purity: 0.00%, (MC - Data) / Data: 0.00%
  Flag: Other, Event Count: 0.00 x 10^6, Efficiency: 0.00%, Purity: 0.00%, (MC - Data) / Data: 0.00%
  Flag: pi+pi-pi+pi-, Event Count: 0.00 x 10^6, Efficiency: 0.00%, Purity: 0.00%, (MC - Data) / Data: 0.00%
  Flag: MC sum, Event Count: 0.00 x 10^6, Efficiency: 0.00%, Purity: 0.00%, (MC - Data) / Data: 0.00%

Results for cut list: ['abs(Kchrec_5) <